# F1 · Definición del proyecto y entorno reproducible

**MCDI500 · Grupo 7 · Sumativa 1**

Esta fase define problema, objetivos, alcance y configuración. No limpia, imputa ni
transforma observaciones. El diagnóstico y las transformaciones corresponden a
F2/notebooks/F2_Preprocesamiento.ipynb.

**Estado:** base inicial preparada con asistencia de Codex. El equipo debe revisar las
decisiones, ejecutar en sus entornos y registrar contribuciones reales. La ejecución
adjunta identifica el entorno de validación; no acredita trabajo de los cuatro
integrantes ni ejecución en sus equipos.


La versión compartida conserva salidas agregadas y ejemplos de conciliación sin nombres ni RUT de proveedores. El original y las tablas individuales se generan localmente; su publicación requiere autorización explícita.

## 1. Problema y contexto

El archivo de Compra Ágil de la unidad ABASTECIMIENTO de la Municipalidad de
Valparaíso reúne órdenes, proveedores, rubros, ítems y cotizaciones. Antes de
describir montos por proveedor, rubro y mes, se debe establecer qué representa cada
fila, qué importes son comparables y qué datos están incompletos.

**Pregunta del mapa V2:** ¿cómo se distribuyen los montos de las órdenes de Compra
Ágil según proveedores, rubros y mes de envío?

**Problema técnico de F1/F2:** construir una base verificable que permita responder
posteriormente esa pregunta sin repetir importes de una misma orden, mezclar monedas
ni transformar datos desconocidos en atributos supuestamente conocidos. F2 determina
la unidad de análisis; no se presupone que cada fila sea una orden o un ítem.


## 2. Objetivos

**General.** Preparar un entorno y un pipeline reproducibles para caracterizar las
órdenes de Compra Ágil observadas en el archivo, conservando procedencia y trazabilidad.

1. Documentar intérprete, dependencias, estructura y SHA-256; verificar todas las rutas
   relativas requeridas.
2. Perfilar el 100 % de las columnas; cuantificar faltantes, repeticiones, monedas,
   estados y cardinalidades antes de transformar.
3. Construir por código una tabla con codigoOC único; conservar todas las órdenes
   observadas y separar pertenencia a rubros del importe de la OC.
4. Justificar conversiones, faltantes, marcado de extremos y variables derivadas;
   registrar cantidades afectadas y parámetros.
5. Probar casos normales, límites y errores; ejecutar F1/F2 desde kernels nuevos y
   conectar las evidencias con informe y commits reales.

Son objetivos de preparación y calidad. Las distribuciones, concentración y
visualizaciones finales se desarrollarán en fases posteriores, según sus guías.


## 3. Alcance, supuestos y exclusiones

- Se trabaja con el archivo adjunto; la cobertura y unidad de compra se confirman en
  F2. No se generaliza a todo Chile ni a años no observados.
- Los atributos de cabecera deben ser constantes por codigoOC; el código se detiene
  si encuentra contradicciones.
- Se propone usar el neto CLP informado por la fuente. Su definición y conversión
  deben contrastarse con el diccionario oficial.
- Se conservan todos los estados; una marca identifica los propuestos para el
  análisis descriptivo. El equipo debe justificar y aprobar el filtro.
- No se afirman pagos, irregularidades, causalidad, estacionalidad anual ni
  comparación con ofertas perdedoras si el archivo solo contiene seleccionadas.
- Los montos por rubro requieren detalle conciliado; no se reparte automáticamente
  el importe total de una OC entre sus rubros.


## 4. Herramientas y fundamento

| Herramienta | Intervención | Justificación |
|---|---|---|
| Python | Funciones y verificaciones | Secuencia repetible |
| pandas | Lectura, perfil y proyección F2 | Claves y agrupaciones comprobables |
| NumPy | Cuantiles y estandarización F2 | Parámetros numéricos explícitos |
| Jupyter | Narrativa y resultados F1/F2 | Evidencia visible junto al código |
| Git / GitHub | Versiones y revisión | Autor, momento y cambio identificables |
| Matplotlib | Figura de calidad | Se genera desde las métricas |
| LaTeX | Informe con tablas derivadas | Evita transcribir cifras |

LaTeX se edita como archivo .tex en JupyterLab. MiKTeX o TeX Live compilan el PDF y
se instalan fuera del entorno virtual de Python.


## 5. Localizar el proyecto

Se busca la raíz desde cualquier subcarpeta.

In [1]:
from pathlib import Path
import sys
import json

inicio = Path.cwd().resolve()
ROOT = next((p for p in (inicio, *inicio.parents) if (p / "proyecto.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Abra Jupyter dentro de proyecto-grupo7-mcdi500.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from F1.src.entorno import cargar_configuracion, verificar_original, reconocer_archivo, registrar_entorno
config = cargar_configuracion(ROOT)
print("Proyecto:", config["proyecto"])
print("Raíz encontrada; las rutas de datos son relativas.")


Proyecto: Compra Ágil: preparación reproducible de órdenes de compra de Valparaíso
Raíz encontrada; las rutas de datos son relativas.


## 6. Entorno observado

Esta evidencia se vuelve a generar al ejecutar en otro equipo. El entorno de
validación asistida y el Windows del grupo se declaran por separado.
requirements.txt conserva el listado previo del equipo; requirements-validacion.txt
describe las dependencias directas usadas en la comprobación inicial. Antes de
entregar, el equipo debe resolver diferencias y probar una reconstrucción limpia.


In [2]:
import pandas as pd
entorno = registrar_entorno()
print("Python:", entorno["python"], "| Sistema:", entorno["sistema"])
print("Intérprete:", entorno["interprete"])
print("Entorno virtual:", entorno["entorno_virtual"])
display(pd.DataFrame(list(entorno["paquetes"].items()), columns=["paquete", "version_observada"]))
(ROOT / "evidencias").mkdir(exist_ok=True)
(ROOT / "evidencias/entorno_F1.json").write_text(json.dumps(entorno, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


Python: 3.12.0 | Sistema: Windows
Intérprete: C:\Users\benja\OneDrive\Escritorio\proyecto-grupo7-mcdi500\.venv\Scripts\python.exe
Entorno virtual: True


,paquete,version_observada
0,numpy,2.5.3
1,pandas,3.0.5
2,matplotlib,3.11.1
3,scikit-learn,1.9.0
4,nbformat,5.11.1
5,nbclient,0.11.0
6,ipykernel,7.3.0
7,jupyterlab,4.6.3


414

## 7. Estructura y archivos clave

Se conserva F1–F4. El original reside en F1/data/raw y los procesados se generan en
F2/data/processed. F1/data/processed pertenecía a la estructura inicial y no recibe
transformaciones. docs/vinculacion_mapa.md relaciona estas rutas con el mapa V2.


In [3]:
rutas = [
    ("F1/data/raw", "Original inmutable"),
    ("F1/notebooks/F1_Definicion.ipynb", "Definición y entorno"),
    ("F1/src/entorno.py", "Reconocimiento y procedencia"),
    ("F1/docs/Mapa_conceptual_Grupo7_V2.pdf", "Mapa corregido"),
    ("F2/notebooks/F2_Preprocesamiento.ipynb", "Pipeline documentado"),
    ("F2/src/preprocesamiento.py", "Funciones de datos"),
    ("F2/tests/test_preprocesamiento.py", "Pruebas"),
    ("F2/data/processed", "Tablas generadas"),
    ("informe/informe.tex", "Informe editable"),
    ("proyecto.json", "Selección y políticas"),
    ("requirements.txt", "Dependencias previas del equipo"),
    (".gitignore", "Exclusiones"),
    (".gitattributes", "Bytes y salidas"),
]
display(pd.DataFrame([{"ruta": r, "funcion": f, "existe": (ROOT/r).exists()} for r,f in rutas]))
assert all((ROOT/r).exists() for r,_ in rutas), "Faltan componentes del proyecto."


,ruta,funcion,existe
0,F1/data/raw,Original inmutable,True
1,F1/notebooks/F1_Definicion.ipynb,Definición y entorno,True
2,F1/src/entorno.py,Reconocimiento y procedencia,True
3,F1/docs/Mapa_conceptual_Grupo7_V2.pdf,Mapa corregido,True
4,F2/notebooks/F2_Preprocesamiento.ipynb,Pipeline documentado,True
5,F2/src/preprocesamiento.py,Funciones de datos,True
6,F2/tests/test_preprocesamiento.py,Pruebas,True
7,F2/data/processed,Tablas generadas,True
8,informe/informe.tex,Informe editable,True
9,proyecto.json,Selección y políticas,True


## 8. Procedencia e integridad

Se copia el adjunto sin volver a guardarlo desde Excel; solo se cambia el nombre a
187402OCCompraAgil.csv. Su SHA-256 coincide con el archivo auditado para el mapa V2:
el sufijo (2) es un nombre de descarga, no evidencia de una versión distinta.
La URL y fecha exactas de descarga deben documentarlas quienes obtuvieron el archivo.


In [4]:
manifiesto = verificar_original(ROOT, config)
display(pd.DataFrame([manifiesto]))
print("Integridad del original: correcta.")


,ruta_relativa,bytes,sha256
0,F1/data/raw/187402OCCompraAgil.csv,21743181,1fcbdec3161500e95a15d2af82df3d007549f2d5da9a6d...


Integridad del original: correcta.


## 9. Reconocimiento sin limpieza

Se usa separador punto y coma, cp1252 y texto para preservar códigos. Los campos
vacíos se reconocen como ausentes al interpretar el CSV; no se les asignan valores.
Las conversiones y el diagnóstico estadístico se desarrollan en F2.


In [5]:
original = reconocer_archivo(ROOT, config)
print("Dimensiones:", original.shape)
print("Columnas:", original.columns.tolist())
display(original.dtypes.astype(str).rename_axis("columna").to_frame("tipo_lectura").head(10))
assert original.columns.tolist() == config["datos"]["columnas_esperadas"]
assert verificar_original(ROOT, config) == manifiesto
print("Original intacto; F1 no exporta datos procesados.")


Dimensiones: (11177, 63)
Columnas: ['codigoOC', 'FechaEnvioOC', 'NombreOC', 'DescripcionOC', 'EstadoOC', 'ProcedenciaOC', 'MonedaOC', 'MontoNetoOC', 'DescuentosOC', 'CargosOC', 'ImpuestosOC', 'MontoTotalOC', 'ImpuestosOC_CLP', 'MontoNetoOC_CLP', 'MetodoPago', 'TipoDespacho', 'Financiamiento', 'UnidadCompra', 'UnidadCompraRUT', 'RegionUnidadCompra', 'entCode', 'Institucion', 'Sector', 'Proveedor', 'ProveedorRUT', 'ActividadProveedor', 'TamanoProveedor', 'RegionProveedor', 'RubroN1', 'RubroN2', 'RubroN3', 'CodigoProductoONU', 'ONUProducto', 'NombreItem', 'DescripcionItem', 'CantidadItem', 'UnidadMedida', 'MonedaItem', 'MontoNetoItem', 'DescuentoItem', 'CargosItem', 'ImpuestoEspecificoItem', 'MontoTotalItem', 'MontoNetoItemCLP', 'CodigoCotizacion', 'NombreCotizacion', 'DescripcionCotizacion', 'FechaPublicacionCotizacion', 'FechaCierreCotizacion', 'MontoDisponible', 'DescripcionProducto', 'CodigoProductoCotizadoONU', 'ProductoCotizado', 'CantidadSolicitada', 'ProveedorCotizacion', 'RUTProv

,tipo_lectura
columna,
codigoOC,string
FechaEnvioOC,string
NombreOC,string
DescripcionOC,string
EstadoOC,string
ProcedenciaOC,string
MonedaOC,string
MontoNetoOC,string
DescuentosOC,string


Original intacto; F1 no exporta datos procesados.


## 10. Versionamiento y paso a F2

Cada integrante usa su identidad real y registra aportes revisados. No se crean
commits vacíos ni se alteran fechas. Las tareas propuestas, aún sin asignar, están en
docs/plan_sumativa1.md. La bitácora registra decisión, motivo, cifras, impacto y estado.

La guía exige salidas visibles. Si se instaló nbstripout, hay que retirar el filtro
de este repositorio y comprobar las salidas en el índice de Git antes del commit.
Ver docs/inicio_gitbash.md.

Continuar en F2/notebooks/F2_Preprocesamiento.ipynb.


In [6]:
print("F1 comprobada: configuración, rutas y procedencia.")
print("Pendiente: revisión del grupo y validación en los demás equipos.")


F1 comprobada: configuración, rutas y procedencia.
Pendiente: revisión del grupo y validación en los demás equipos.
